  >Guilherme da Silva ferreira Batista

In [ ]:
"""
Auditoria de Orçamentos Corporativos
Disciplina: Python Avançado
Conceitos aplicados: Recursão, Decorators, *args, **kwargs
"""

import time


# =============================================================================
# 1. DECORATOR @auditor
# =============================================================================

def auditor(funcao):
    """
    Decorator responsável por:
    - Exibir cabeçalho de auditoria antes da execução
    - Registrar os argumentos recebidos
    - Medir e exibir o tempo total de execução
    """
    def wrapper(*args, **kwargs):
        # Cabeçalho da auditoria
        print("=" * 60)
        print("       SISTEMA DE AUDITORIA - INÍCIO DO CÁLCULO")
        print("=" * 60)

        # Exibe os departamentos ignorados (args), se houver
        if args:
            # O primeiro argumento é o dicionário da empresa; os demais são os departamentos ignorados
            estrutura = args[0]
            ignorados = args[1:]
            print(f"  Estrutura recebida: {type(estrutura).__name__}")
            if ignorados:
                print(f"  Departamentos ignorados: {ignorados}")
            else:
                print("  Departamentos ignorados: nenhum")

        # Exibe os parâmetros nomeados de conversão (kwargs), se houver
        if kwargs:
            print(f"  Parâmetros de conversão: {kwargs}")
        else:
            print("  Conversão de moeda: não aplicada")

        print("-" * 60)

        # Marca o tempo de início
        inicio = time.time()

        # Chama a função original
        resultado = funcao(*args, **kwargs)

        # Calcula o tempo de execução
        fim = time.time()
        tempo_total = fim - inicio

        print("-" * 60)
        print(f"  Tempo de processamento: {tempo_total:.6f} segundos")
        print("=" * 60)

        return resultado

    return wrapper


# =============================================================================
# 2. ESTRUTURA DE DADOS - Hierarquia Corporativa (dicionário aninhado)
# =============================================================================

empresa = {
    "Matriz": {
        "TI": {
            "Infraestrutura": 150_000,
            "Desenvolvimento": {
                "Backend": 200_000,
                "Frontend": 180_000,
                "QA": 90_000
            },
            "Segurança": 120_000
        },
        "RH": {
            "Recrutamento": 80_000,
            "Treinamento": 60_000,
            "Benefícios": 110_000
        },
        "Financeiro": {
            "Contabilidade": 95_000,
            "Controladoria": 130_000,
            "Tesouraria": 75_000
        }
    },
    "Filial_SP": {
        "Vendas": {
            "Comercial": 220_000,
            "Marketing": 160_000,
            "CRM": 50_000
        },
        "Operações": {
            "Logística": 140_000,
            "Estoque": 85_000
        }
    },
    "Filial_RJ": {
        "Vendas": {
            "Comercial": 190_000,
            "Marketing": 130_000
        },
        "Suporte": 70_000
    }
}


# =============================================================================
# 3. FUNÇÃO RECURSIVA calcular_orcamento COM @auditor
# =============================================================================

@auditor
def calcular_orcamento(estrutura, *args, **kwargs):
    """
    Calcula recursivamente o orçamento total de uma estrutura corporativa.

    Parâmetros:
        estrutura (dict): Dicionário aninhado representando a empresa.
        *args: Nomes de departamentos a serem IGNORADOS no cálculo.
        **kwargs: Parâmetros opcionais de conversão de moeda:
                  - moeda_destino (str): Sigla da moeda alvo (ex: "BRL")
                  - taxa_cambio (float): Fator de conversão a ser aplicado

    Retorna:
        float: Soma total dos orçamentos com conversão aplicada (se informada).
    """

    def _somar_recursivo(no, ignorados):
        """
        Função auxiliar interna que percorre o dicionário de forma recursiva.
        Separada para que o decorator @auditor só envolva a chamada principal.
        """
        total = 0

        # Se o nó for um número (folha da árvore), retorna ele diretamente
        if isinstance(no, (int, float)):
            return no

        # Se o nó for um dicionário, percorre cada chave
        if isinstance(no, dict):
            for chave, valor in no.items():

                # Verifica se o departamento atual deve ser ignorado
                if chave in ignorados:
                    print(f"  [IGNORADO] Departamento '{chave}' e seus subsetores foram excluídos.")
                    continue  # Pula esse ramo inteiro

                # Chama a si mesma para descer mais um nível na hierarquia
                total += _somar_recursivo(valor, ignorados)

        return total

    # Coleta os departamentos ignorados vindos de *args
    departamentos_ignorados = set(args)

    # Executa a recursão a partir da raiz do dicionário
    total_base = _somar_recursivo(estrutura, departamentos_ignorados)

    # Aplica conversão de moeda se os kwargs necessários forem fornecidos
    taxa = kwargs.get("taxa_cambio", 1.0)
    moeda = kwargs.get("moeda_destino", "USD")

    total_convertido = total_base * taxa

    print(f"\n  Orçamento base (USD): $ {total_base:,.2f}")
    print(f"  Taxa de câmbio aplicada ({moeda}): {taxa}")
    print(f"  Total convertido ({moeda}): R$ {total_convertido:,.2f}")

    return total_convertido


# =============================================================================
# 4. EXECUÇÃO - Chamadas de exemplo
# =============================================================================

if __name__ == "__main__":

    print("\n\n>>> CÁLCULO 1: Orçamento completo da empresa em BRL\n")
    total_1 = calcular_orcamento(
        empresa,
        moeda_destino="BRL",
        taxa_cambio=5.20
    )
    print(f"\n  RESULTADO FINAL: R$ {total_1:,.2f}\n")


    print("\n\n>>> CÁLCULO 2: Ignorando 'Vendas' e 'RH', convertendo para BRL\n")
    total_2 = calcular_orcamento(
        empresa,
        "Vendas",
        "RH",
        moeda_destino="BRL",
        taxa_cambio=5.20
    )
    print(f"\n  RESULTADO FINAL: R$ {total_2:,.2f}\n")


    print("\n\n>>> CÁLCULO 3: Apenas TI e Financeiro (ignorando os demais), sem conversão\n")
    total_3 = calcular_orcamento(
        empresa,
        "Vendas",
        "RH",
        "Operações",
        "Suporte",
        "Filial_RJ"
    )
    print(f"\n  RESULTADO FINAL: $ {total_3:,.2f}\n")